# V23 — Copper WTI | One-shot gap-filling (Kaggle Run All)

**Input:** `Copper, Aluminium, Tin, WTI` (4 variables). **Target:** next-session Copper; 22 historical sessions. This notebook **does not search SA again**. It reuses exact V22 SA-selected hyperparameters (`47, 127, lr=5.0478806754686805e-05`) recorded in the uploaded V22 result ZIP.

**Part A:** Fill *only* missing V22 SA comparisons at 92.5% and 95%, `rolling_756` and `recency_weighted` (5 model variants including Naive). **Part B:** 80/20 test-period walk-forward, refit every 63 sessions, 5 seeds (41–45), matching date coordinates for selected `lstm_residual`, `gru_residual`, `reopt_sa_lstm`, plus non-fitted Naive. **Part C:** paired loss differences, descriptive DM-HAC, plots, ZIP.

**Kaggle input:** Add `dataset_9_commodities_daily.csv` as a Kaggle dataset before Run All. The source V22 ZIP is *not required* because SA HP values are embedded and audited. Use GPU. Full run may be compute-intensive; checkpoint CSV saves every completed job. No fabricated metrics.

**Scientific scope:** This completes specified missing *WTI* experiments; it does **not** establish equal-budget tuning or 2018–2026 historical walk-forward equivalence with V20. Feature effects remain observational when tuning or training setups differ.

In [ ]:
from pathlib import Path
import gc, json, hashlib, time, zipfile, warnings, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import norm
import tensorflow as tf

TARGET='Copper'
FEATURES=['Copper','Aluminium','Tin','WTI']
FEATURE_SETS={'wti_replacement_with_copper': FEATURES}
FEATURE_KEY='wti_replacement_with_copper'
LOOKBACK=22
SEED=42
BATCH_SIZE=64
RATIOS=(0.925,0.95)
SCHEMES=('rolling_756','recency_weighted')
WF_RATIO=0.80
WF_STEP=63
WF_SEEDS=(41,42,43,44,45)
WF_MODELS=('lstm_residual','gru_residual','reopt_sa_lstm')
# SA HP are actual artifacts from V22, not reselected on test.
SA_HP={'units_1':47, 'units_2':127, 'learning_rate':5.0478806754686805e-05}
PAPER_HP={'units_1':39,'units_2':111,'learning_rate':5e-5}
# Use V22 training recipes for holdout; WF uses a shorter common retraining schedule.
HOLDOUT_EPOCHS=100
HOLDOUT_PATIENCE=9
WF_EPOCHS=45
WF_PATIENCE=7
RESUME=True  # For interrupted Kaggle runs, attach exported CSV or retain /kaggle/working within the same session.
OUT=Path('/kaggle/working/copper_v23_gapfill') if Path('/kaggle/working').exists() else Path('copper_v23_gapfill')
OUT.mkdir(parents=True,exist_ok=True)
CANDIDATES=[Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
            Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
            Path('dataset_9_commodities_daily.csv'),Path('data/processed/dataset_9_commodities_daily.csv')]
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError:pass
print('TensorFlow',tf.__version__,'GPUs',tf.config.list_physical_devices('GPU'))
print('Outputs:',OUT.resolve())

In [ ]:
data_path=next((p for p in CANDIDATES if p.is_file()),None)
if data_path is None and Path('/kaggle/input').exists():
    hits=list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(hits)==1:data_path=hits[0]
    elif len(hits)>1:raise RuntimeError(f'Multiple dataset candidates: {hits}. Specify path to ensure correct source.')
if data_path is None:raise FileNotFoundError('Attach dataset_9_commodities_daily.csv in Kaggle Add Input.')
raw=pd.read_csv(data_path)
raw['Date']=pd.to_datetime(raw['Date'],errors='raise')
if raw.Date.duplicated().any():raise ValueError('Duplicate Date in input')
for c in FEATURES:raw[c]=pd.to_numeric(raw[c],errors='raise')
raw=raw.sort_values('Date').reset_index(drop=True)
if raw[FEATURES].isna().any().any():raise ValueError('Missing in V23 required features; verify alignment to V22.')
if not np.isfinite(raw[FEATURES].to_numpy()).all() or (raw[FEATURES]<=0).any().any():raise ValueError('Invalid prices')
data=raw.copy()
SHA=hashlib.sha256(data_path.read_bytes()).hexdigest()
EXPECTED_SHA='9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600'
if SHA!=EXPECTED_SHA:raise ValueError(f'Dataset fingerprint differs from historical V20/V21: {SHA}. Do not merge experiments with mismatched dataset!')
assert len(data)==2710, f'Unexpected row count {len(data)}'
print('Dataset verified',len(data),'rows',str(data.Date.iloc[0].date()),'to',str(data.Date.iloc[-1].date()),SHA)

In [ ]:
CFG={'earlystop':True,'epochs':HOLDOUT_EPOCHS,'search_epochs':15}

In [ ]:
def make_context(frame, ratio, scheme='expanding', feature_set='wti_replacement_with_copper', val_fraction=0.10):
    n = len(frame)
    split = int(n * ratio)
    val_start = int(split * (1.0 - val_fraction))
    fit_start = 0 if scheme != 'rolling_756' else max(0, val_start - 756)
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 25 < val_start < split < n
    assert val_start - fit_start > LOOKBACK + 10
    # Train-only scaling: no future statistics leak into scaler.
    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])
    x = xsc.transform(frame[cols]).astype('float32')
    y = ysc.transform(frame[[TARGET]]).ravel().astype('float32')
    prev = np.r_[np.nan, y[:-1]]
    def block(start, end, restrict_train=False):
        indices = np.arange(max(start, LOOKBACK), end)
        if restrict_train:
            indices = indices[indices-LOOKBACK >= fit_start]
        X = np.stack([x[t-LOOKBACK:t] for t in indices]).astype('float32')
        return {'idx': indices, 'X': X, 'y': y[indices],
                'prev': prev[indices].astype('float32'),
                'date': frame.Date.iloc[indices].to_numpy()}
    tr = block(fit_start, val_start, True)
    va = block(val_start, split)
    te = block(split, n)
    assert tr['idx'][-1] < va['idx'][0] < te['idx'][0]
    assert xsc.n_samples_seen_ == val_start - fit_start
    assert len(tr['X']) > 0 and tr['X'].shape[1:] == (LOOKBACK, len(cols))
    # Last X timestamp is always t-1. Verify Copper only if included.
    if 'Copper' in cols:
        copper_col = cols.index('Copper')
        copper_raw_in_x = xsc.inverse_transform(x[tr['idx']-1])[:, copper_col]
        copper_true = frame.iloc[tr['idx']-1]['Copper'].to_numpy()
        assert np.allclose(copper_raw_in_x, copper_true, atol=1e-4)
    return {'train': tr, 'val': va, 'test': te, 'xsc': xsc, 'ysc': ysc,
            'features': cols, 'feature_set': feature_set, 'horizon': 1,
            'ratio': float(ratio), 'scheme': scheme,
            'fit_start': fit_start, 'val_start': val_start, 'test_start': split}

In [ ]:
PAPER_SA_LSTM_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}

def build_baseline_lstm(input_shape, hp=None):
    """Mô hình LSTM chuẩn không tối ưu SA (Protocol V20 baseline)."""
    model = tf.keras.Sequential(name='Baseline_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

def build_paper_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM với siêu tham số công bố trong bài báo Chen et al. (2023)."""
    model = tf.keras.Sequential(name='Paper_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='mse')
    return model

def build_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM nhận siêu tham số linh hoạt do SA tìm kiếm (chỉ H=1, HP tìm mới riêng cho từng feature set)."""
    h = hp or PAPER_SA_LSTM_HP
    u1 = int(h.get('units_1', 39))
    u2 = int(h.get('units_2', 111))
    lr = float(h.get('learning_rate', 5e-5))

    model = tf.keras.Sequential(name='Reoptimized_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(u1, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(u2, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse')
    return model

def build_baseline_rnn(input_shape, hp=None):
    """SimpleRNN baseline with two recurrent layers and comparable head."""
    model = tf.keras.Sequential(name='Baseline_RNN')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.SimpleRNN(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.SimpleRNN(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
def reset_keras(seed=SEED):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(int(seed))

def sample_weights(length, scheme, half_life=252):
    if scheme != 'recency_weighted': return None
    ages = np.arange(length - 1, -1, -1, dtype='float64')
    w = np.exp(-np.log(2) * ages / half_life)
    return (w / w.mean()).astype('float32')

def evaluate_predictions(ctx, model_name, p_val, p_test, seed=SEED, extras=None):
    ysc = ctx['ysc']; va = ctx['val']; te = ctx['test']
    def original(z): return ysc.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel()

    # 1. Dịch ngược về giá gốc (USD)
    y_true = original(te['y'])
    pred = original(p_test)
    base = original(te['prev'])

    delta_pred = pred - base
    delta_true = y_true - base

    # 2. Tính các độ đo trên giá gốc (RAW - USD) khớp 100% với V20
    mae = float(mean_absolute_error(y_true, pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
    mse = float(mean_squared_error(y_true, pred))  # DỊCH NGƯỢC LẠI GIÁ GỐC (USD^2) - KHỚP V20
    base_mae = float(mean_absolute_error(y_true, base))
    improvement = float(100.0 * (base_mae - mae) / base_mae) if base_mae > 0 else 0.0

    # 3. Tính các độ đo trên thang đo chuẩn hóa (SCALED [0, 1])
    val_mse_scaled = float(mean_squared_error(va['y'], p_val))
    mse_scaled = float(mean_squared_error(te['y'], p_test)) if p_test is not None else np.nan

    # 4. Tính Directional Accuracy và các thống kê kiểm toán phần dư
    dir_acc = float(100.0 * np.mean(np.sign(delta_pred) == np.sign(delta_true))) if model_name != 'naive' else np.nan
    delta_pred_std = float(np.std(delta_pred))
    delta_true_std = float(np.std(delta_true))
    delta_pred_mean_abs = float(np.mean(np.abs(delta_pred)))
    pred_delta_corr = float(np.corrcoef(delta_pred, delta_true)[0, 1]) if delta_pred_std > 1e-12 else np.nan

    row = {
        'model': model_name,
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'scheme': ctx['scheme'],
        'ratio': ctx['ratio'],
        'seed': int(seed),
        'n_test': len(y_true),
        'mae': mae,
        'rmse': rmse,
        'mse': mse,                        # GIÁ GỐC DỊCH NGƯỢC (USD^2) - KHỚP V20
        'mse_scaled': mse_scaled,          # SCALED TRÊN TẬP TEST [0, 1]
        'mae_naive': base_mae,
        'mae_improvement_pct': improvement,
        'val_mse_scaled': val_mse_scaled,  # SCALED TRÊN TẬP VALIDATION [0, 1] - KHỚP V20
        'directional_accuracy_pct': dir_acc,
        'delta_pred_std_raw': delta_pred_std,
        'delta_true_std_raw': delta_true_std,
        'delta_pred_mean_abs_raw': delta_pred_mean_abs,
        'pred_delta_corr': pred_delta_corr,
        'test_start': str(pd.Timestamp(te['date'][0]).date()),
        'fit_start': str(data.Date.iloc[ctx['fit_start']].date())
    }
    if extras: row.update(extras)

    predictions = pd.DataFrame({
        'Date': te['date'],
        'ratio': ctx['ratio'],
        'scheme': ctx['scheme'],
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'model': model_name,
        'seed': int(seed),
        'actual': y_true,
        'pred': pred,
        'naive': base,
        'pred_delta': delta_pred,
        'actual_delta': delta_true
    })
    return row, predictions

def fit_predict_model(builder, ctx, hp=None, seed=SEED, epochs=None, patience=None, is_search=False, skip_test=False):
    reset_keras(seed)
    tr, va, te = ctx['train'], ctx['val'], ctx['test']
    model = builder(tr['X'].shape[1:], hp)

    cb = []
    if CFG['earlystop']:
        pat = patience if patience is not None else (4 if is_search else 9)
        cb.append(tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat, restore_best_weights=True))

    ep = epochs if epochs is not None else (CFG['search_epochs'] if is_search else CFG['epochs'])
    weights = sample_weights(len(tr['idx']), ctx['scheme'])

    start_t = time.perf_counter()
    hist = model.fit(
        tr['X'], tr['y'],
        validation_data=(va['X'], va['y']),
        epochs=ep,
        batch_size=BATCH_SIZE,
        sample_weight=weights,
        callbacks=cb,
        shuffle=False,
        verbose=0
    )
    fit_sec = time.perf_counter() - start_t

    pv = model.predict(va['X'], verbose=0).ravel()
    pt = model.predict(te['X'], verbose=0).ravel() if not skip_test else None
    val_mse = float(mean_squared_error(va['y'], pv))
    epochs_run = len(hist.history['loss'])

    del model
    reset_keras(seed)
    return {'val_pred': pv, 'test_pred': pt, 'val_mse': val_mse, 'seconds': fit_sec, 'epochs_run': epochs_run}

In [ ]:
def build_net(arch, shape):
    inp=tf.keras.Input(shape=shape)
    z=inp
    if arch in ('cnn_lstm','cnn_gru'):
        z=tf.keras.layers.Conv1D(32,3,padding='causal',activation='relu')(z)
        z=tf.keras.layers.MaxPooling1D(2)(z)
    if arch in ('lstm','cnn_lstm'):
        n1,n2=(64,32) if arch=='lstm' else (48,24)
        z=tf.keras.layers.LSTM(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.LSTM(n2)(z)
    elif arch in ('gru','cnn_gru'):
        n1,n2=(64,32) if arch=='gru' else (48,24)
        z=tf.keras.layers.GRU(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.GRU(n2)(z)
    else: raise ValueError(arch)
    z=tf.keras.layers.Dropout(.2)(z)
    z=tf.keras.layers.Dense(16,activation='relu')(z)
    out=tf.keras.layers.Dense(1,activation='linear')(z)
    model=tf.keras.Model(inp,out,name=arch)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
    return model

## Part A — Fill missing SA-LSTM holdout configurations

4 settings (`92.5%`, `95%`) × (`rolling_756`, `recency_weighted`) × 5 models. The four fitted models are rebuilt with the V22 recipes, including the previously selected V22 SA hyperparameters; the Naive baseline has no training. No SA candidate search is performed here. Checkpointing is keyed by dataset SHA and hyperparameters.

In [ ]:
RUN_HASH=hashlib.sha256(json.dumps({'data_sha':SHA,'features':FEATURES,'ratio':RATIOS,'schemes':SCHEMES,'sa_hp':SA_HP,'epochs':HOLDOUT_EPOCHS,'patience':HOLDOUT_PATIENCE,'seed':SEED,'tensorflow':tf.__version__},sort_keys=True).encode()).hexdigest()[:16]
PART_A=OUT/'holdout_missing_results.csv'
PART_A_PRED=OUT/'holdout_missing_predictions.csv'
A_MODELS=[('naive',None),('baseline_lstm',build_baseline_lstm),('baseline_rnn',build_baseline_rnn),('paper_sa_lstm',build_paper_sa_lstm),('reopt_sa_lstm',build_sa_lstm)]

def csv_append(path,row):
    pd.DataFrame([row]).to_csv(path,mode='a',header=not path.exists(),index=False)

def check_and_load(path,expected):
    if not path.exists():return pd.DataFrame()
    df=pd.read_csv(path)
    if not df.empty and ('run_hash' not in df or not df['run_hash'].eq(expected).all()):
        raise RuntimeError(f'Incompatible checkpoint at {path}. Remove or move before Run All.')
    return df

if not RESUME:
    for path in (PART_A,PART_A_PRED):path.unlink(missing_ok=True)
a_old=check_and_load(PART_A,RUN_HASH)
a_done=set(zip(a_old.ratio.round(3),a_old.scheme,a_old.model)) if not a_old.empty else set()
a_jobs=[(ratio,scheme,name,builder) for ratio in RATIOS for scheme in SCHEMES for name,builder in A_MODELS]
for i,(ratio,scheme,name,builder) in enumerate(a_jobs,1):
    key=(round(ratio,3),scheme,name)
    if key in a_done:continue
    ctx=make_context(data,ratio,scheme,FEATURE_KEY)
    if name=='naive':
        pv=ctx['val']['prev'];pt=ctx['test']['prev'];meta={'epochs_run':0,'fit_seconds':0,'hp_config':'none'}
    else:
        hp=SA_HP if name=='reopt_sa_lstm' else PAPER_HP if name=='paper_sa_lstm' else None
        fit=fit_predict_model(builder,ctx,hp=hp,seed=SEED,epochs=HOLDOUT_EPOCHS,patience=HOLDOUT_PATIENCE)
        pv=fit['val_pred'];pt=fit['test_pred'];meta={'epochs_run':fit['epochs_run'],'fit_seconds':fit['seconds'],'hp_config':json.dumps(hp or {})}
    row,pred=evaluate_predictions(ctx,name,pv,pt,seed=SEED,extras=meta)
    row.update({'run_hash':RUN_HASH,'source':'V23_gapfill_holdout'})
    pred['run_hash']=RUN_HASH;pred['source']='V23_gapfill_holdout'
    csv_append(PART_A,row)
    pred.to_csv(PART_A_PRED,mode='a',header=not PART_A_PRED.exists(),index=False)
    a_done.add(key)
    print(f'A {i}/{len(a_jobs)} {ratio} {scheme} {name} MAE={row["mae"]:.6f}',flush=True)
assert len(a_done)==len(a_jobs),f'Only {len(a_done)}/{len(a_jobs)} completed'
a=pd.read_csv(PART_A)
print('Completed PART A',len(a),'evaluations')
display(a.pivot_table(index=['ratio','model'],columns='scheme',values='mae'))

## Part B — Aligned 80/20 walk-forward, five seeds

The first test prediction begins exactly at the 80/20 chronological boundary (542 test observations). The next 63-step block is forecast from a **single model fitted at its origin**, without fitting on any outcomes within that block. All models use the same fold origins, timestamps, seeds, and inputs. Scalers fit only on earlier training subwindows; validation precedes every forecast block. The SA parameters are fixed from V22 and are not searched during walk-forward. The three models provide a focused robustness check, not an exhaustive every-architecture comparison.

In [ ]:
def fold_context(origin,end):
    val_start=int(origin*0.9)
    assert LOOKBACK+30<val_start<origin<end<=len(data)
    xsc=MinMaxScaler().fit(data.iloc[:val_start][FEATURES])
    ysc=MinMaxScaler().fit(data.iloc[:val_start][['Copper']])
    x=xsc.transform(data[FEATURES]).astype('float32')
    y=ysc.transform(data[['Copper']]).ravel().astype('float32')
    prev=np.r_[np.nan,y[:-1]]
    def block(first,last):
        ids=np.arange(max(first,LOOKBACK),last)
        return {'idx':ids,'X':np.stack([x[t-LOOKBACK:t] for t in ids]),'y':y[ids],
                'prev':prev[ids].astype('float32'),'date':data.Date.iloc[ids].to_numpy()}
    tr=block(LOOKBACK,val_start);va=block(val_start,origin);te=block(origin,end)
    assert tr['idx'][-1]<va['idx'][0]<te['idx'][0]
    assert np.allclose(te['X'][:,-1,0],te['prev'],atol=2e-6)
    return {'train':tr,'val':va,'test':te,'xsc':xsc,'ysc':ysc,'features':FEATURES,
            'feature_set':FEATURE_KEY,'scheme':'expanding','ratio':origin/len(data),
            'fit_start':0,'val_start':val_start,'test_start':origin}

def fit_wf(ctx,name,seed):
    reset_keras(seed)
    tr,va,te=[ctx[k] for k in ['train','val','test']]
    if name=='reopt_sa_lstm':
        model=build_sa_lstm(tr['X'].shape[1:],SA_HP)
        train_y=tr['y'];val_y=va['y']
    else:
        model=build_net(name.split('_')[0],tr['X'].shape[1:])
        train_y=tr['y']-tr['prev'];val_y=va['y']-va['prev']
    cb=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=WF_PATIENCE,restore_best_weights=True)]
    t0=time.monotonic()
    hist=model.fit(tr['X'],train_y,validation_data=(va['X'],val_y),epochs=WF_EPOCHS,
                   batch_size=BATCH_SIZE,callbacks=cb,shuffle=False,verbose=0)
    pv=model.predict(va['X'],verbose=0).ravel()
    pt=model.predict(te['X'],verbose=0).ravel()
    if name.endswith('_residual'):
        pv+=va['prev'];pt+=te['prev']
    meta={'epochs_run':len(hist.history['loss']),'fit_seconds':time.monotonic()-t0,
          'hp_config':json.dumps(SA_HP) if name=='reopt_sa_lstm' else 'fixed_V22_arch'}
    del model;reset_keras(seed)
    return pv,pt,meta

WF_HASH=hashlib.sha256(json.dumps({'sha':SHA,'features':FEATURES,'ratio':WF_RATIO,'step':WF_STEP,
    'seeds':WF_SEEDS,'models':WF_MODELS,'sa_hp':SA_HP,'epochs':WF_EPOCHS,'patience':WF_PATIENCE,
    'tensorflow':tf.__version__},sort_keys=True).encode()).hexdigest()[:16]
WF_METRICS=OUT/'wf_fold_metrics.csv';WF_PREDS=OUT/'wf_predictions.csv'
if not RESUME:
    for p in (WF_METRICS,WF_PREDS):p.unlink(missing_ok=True)
wf_old=check_and_load(WF_METRICS,WF_HASH)
wf_done=set(zip(wf_old.fold_start.astype(int),wf_old.seed.astype(int),wf_old.model)) if not wf_old.empty else set()
first=int(len(data)*WF_RATIO)
folds=[(start,min(start+WF_STEP,len(data))) for start in range(first,len(data),WF_STEP)]
print('WF:',len(folds),'folds,',len(WF_SEEDS),'seeds,',len(WF_MODELS),'models =',len(folds)*len(WF_SEEDS)*len(WF_MODELS),'fits')
for fi,(start,end) in enumerate(folds,1):
    ctx=fold_context(start,end)
    for seed in WF_SEEDS:
        for name in WF_MODELS:
            key=(start,seed,name)
            if key in wf_done:continue
            pv,pt,meta=fit_wf(ctx,name,seed)
            row,pred=evaluate_predictions(ctx,name,pv,pt,seed=seed,extras=meta)
            row.update({'fold_start':start,'fold_end':end,'fold_number':fi,'run_hash':WF_HASH,'source':'V23_walk_forward'})
            pred['fold_start']=start;pred['fold_end']=end;pred['fold_number']=fi
            pred['run_hash']=WF_HASH;pred['source']='V23_walk_forward'
            csv_append(WF_METRICS,row)
            pred.to_csv(WF_PREDS,mode='a',header=not WF_PREDS.exists(),index=False)
            wf_done.add(key)
            print(f'WF {fi}/{len(folds)} seed={seed} {name} MAE={row["mae"]:.6f}',flush=True)
assert len(wf_done)==len(folds)*len(WF_SEEDS)*len(WF_MODELS)
wfp=pd.read_csv(WF_PREDS,parse_dates=['Date'])
assert wfp.groupby(['seed','model']).size().eq(len(data)-first).all()
for (_, _),g in wfp.groupby(['seed','model']):
    assert g.Date.is_unique and g.Date.sort_values().reset_index(drop=True).equals(data.Date.iloc[first:].reset_index(drop=True))
print('WF forecasts validated, same timestamps:',len(data)-first)

## Part C — Recompute summary from raw predictions, paired tests, plots, and ZIP

Ranks exclude Naive but Naive remains a reference. P-values are exploratory and unadjusted for multiple comparisons. Better MAE on one holdout does not itself demonstrate statistically reliable gains.

In [ ]:
def dm_hac(loss_difference,lags=5):
    d=np.asarray(loss_difference,dtype=float)
    if len(d)<10 or not np.isfinite(d).all():return {'mean_loss_diff':np.nan,'dm_z':np.nan,'p_value':np.nan}
    x=d-d.mean();n=len(x);var=np.dot(x,x)/n
    for lag in range(1,min(lags,n-1)+1):
        cov=np.dot(x[lag:],x[:-lag])/n
        var+=2*(1-lag/(lags+1))*cov
    if var<=0:return {'mean_loss_diff':float(d.mean()),'dm_z':np.nan,'p_value':np.nan}
    z=float(d.mean()/np.sqrt(var/n))
    return {'mean_loss_diff':float(d.mean()),'dm_z':z,'p_value':float(2*norm.sf(abs(z)))}

summ=[];dm=[]
for (seed,name),g in wfp.groupby(['seed','model']):
    g=g.sort_values('Date')
    ae=np.abs(g.actual.to_numpy()-g.pred.to_numpy())
    an=np.abs(g.actual.to_numpy()-g.naive.to_numpy())
    summ.append({'seed':seed,'model':name,'n':len(g),'mae':float(ae.mean()),'rmse':float(np.sqrt(np.mean((g.actual-g.pred)**2))),
                 'naive_mae':float(an.mean()),'skill_vs_naive_pct':float(100*(1-ae.mean()/an.mean()))})
    dm.append({'seed':seed,'model':name,'test':'absolute_error_model_minus_naive','hac_lags':5,**dm_hac(ae-an,5)})
s=pd.DataFrame(summ).sort_values(['seed','mae']); d=pd.DataFrame(dm)
s.to_csv(OUT/'wf_seed_summary.csv',index=False);d.to_csv(OUT/'wf_dm_hac_by_seed.csv',index=False)
rank=a[a.model!='naive'].sort_values(['ratio','scheme','mae']).copy()
rank['rank_in_split_scheme']=rank.groupby(['ratio','scheme']).cumcount()+1
rank.to_csv(OUT/'holdout_fitted_rankings.csv',index=False)
display(rank[['ratio','scheme','rank_in_split_scheme','model','mae','rmse']]);display(s);display(d)
fig,ax=plt.subplots(figsize=(10,5))
for name,g in a.groupby('model'):
    ax.plot([f'{r:.3f} {sc}' for r,sc in zip(g.ratio,g.scheme)],g.mae,marker='o',label=name)
ax.set_ylabel('Holdout MAE');ax.set_title('V23 missing SA benchmark configurations');ax.tick_params(axis='x',rotation=35);ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/'holdout_missing_mae.png',dpi=150);plt.close(fig)
fig,ax=plt.subplots(figsize=(10,5))
for name,g in s.groupby('model'):
    g=g.sort_values('seed');ax.plot(g.seed,g.mae,marker='o',label=name)
ax.axhline(s.naive_mae.iloc[0],linestyle='--',color='gray',label='Naive')
ax.set(xlabel='Seed',ylabel='MAE',title='Aligned 80/20 walk-forward, 5 seeds');ax.legend();fig.tight_layout();fig.savefig(OUT/'wf_mae_by_seed.png',dpi=150);plt.close(fig)
metadata={'source_dataset':str(data_path),'sha256':SHA,'rows':len(data),'dates':[str(data.Date.iloc[0].date()),str(data.Date.iloc[-1].date())],
 'features':FEATURES,'lookback':LOOKBACK,'horizon':1,'missing_holdout_ratios':RATIOS,'missing_holdout_schemes':SCHEMES,
 'holdout_epochs':HOLDOUT_EPOCHS,'holdout_patience':HOLDOUT_PATIENCE,'sa_hp_from_v22':SA_HP,
 'walk_forward_start_ratio':WF_RATIO,'walk_forward_start_date':str(data.Date.iloc[first].date()),
 'walk_forward_step':WF_STEP,'walk_forward_seeds':WF_SEEDS,'walk_forward_models':WF_MODELS,
 'walk_forward_epochs':WF_EPOCHS,'walk_forward_patience':WF_PATIENCE,
 'walk_forward_method':'origin-refit, full-history expanding; internal tail validation 10%; no future labels at fit time',
 'scope':'WTI-only; selected models only; not an exhaustive matched-budget assessment',
 'part_a_jobs':len(a),'part_b_fits':len(wf_done),'framework':tf.__version__}
(OUT/'experiment_manifest.json').write_text(json.dumps(metadata,indent=2,default=str),encoding='utf-8')
(OUT/'README_RESULTS.md').write_text('# V23 Experimental gap filling\n\n' + json.dumps(metadata,indent=2,default=str) + '\n\nFiles: holdout_missing_results.csv, holdout_missing_predictions.csv, holdout_fitted_rankings.csv, wf_fold_metrics.csv, wf_predictions.csv, wf_seed_summary.csv, wf_dm_hac_by_seed.csv.\n',encoding='utf-8')
ZIP_PATH=OUT.parent/'copper_v23_gapfill_results.zip'
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=OUT.name+'/'+f.name)
print('RUN ALL COMPLETE:',ZIP_PATH.resolve())
print('Contains:',sorted(x.name for x in OUT.iterdir()))